# 03B Real Business Cycle (RBC) Models: Solution

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](LICENSE) 
[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [ ]:
# === Environment Setup ===
import os
import sys
import math
import time
import random
import json
import textwrap
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from IPython.display import display, Markdown, Image

# Add scripts directory to path to import utility modules
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), '..', 'scripts'))
try:
    from macro_utils import solve_qz
except ImportError:
    print("Warning: macro_utils not found. Please ensure scripts/macro_utils.py exists.")

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6), 'figure.dpi': 130,
                     'axes.titlesize': 'x-large', 'axes.labelsize': 'large',
                     'xtick.labelsize': 'medium', 'ytick.labelsize': 'medium'})
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(suppress=True, linewidth=140, precision=4)
warnings.filterwarnings('ignore', category=FutureWarning)


Environment initialized for Advanced RBC model analysis.


# The Lens: Solving and Simulating RBC Dynamics

**What problem are we solving?**
Once the RBC equilibrium conditions are established, we need a practical way to compute decision rules and simulate the economy under shocks. The key challenge is transforming a nonlinear system into a solvable linear representation and then interpreting the resulting dynamics.

**Why this method?**
Log-linearization and the QZ (generalized Schur) decomposition provide a stable, transparent way to solve linear rational expectations models. They let us compute policy and transition matrices that can be simulated under surprise or news shocks.

### Learning Objectives
* **Log-linearize** the RBC system around the steady state.
* **Solve** the linearized model using QZ decomposition (Blanchard–Kahn conditions).
* **Prepare** solution objects for simulation in the dynamics notebook.

### Prerequisites
* **`04-Macro-Models/03A_RBC_Model_Foundations.ipynb`**: Equilibrium conditions and planner setup.
* **`02-Numerical-Methods/01_Linear_Algebra.ipynb`**: Eigenvalues and matrix factorization.
* **`02-Numerical-Methods/05_Optimization.ipynb`**: Solving nonlinear systems.

# Table of Contents
* [The Lens: Solving and Simulating RBC Dynamics](#The-Lens:-Solving-and-Simulating-RBC-Dynamics)
* [1. Solving the Model: Log-Linearization and QZ Decomposition](#1-solving-the-model:-log-linearization-and-qz-decomposition)
* [Summary](#summary)

## 1. Solving the Model: Log-Linearization and QZ Decomposition

Since the system of equations is non-linear, we cannot solve it analytically. Instead, we approximate the solution by **log-linearizing** around the deterministic steady state. This yields a system of linear difference equations of the form:

$$ A E_t[x_{t+1}] = B x_t + C \epsilon_{t+1} $$

To ensure numerical stability and avoid singularity issues with static variables (like $w_t, r_t, y_t$), we solve a **reduced-form** system involving only the core dynamic variables: capital ($k$), productivity ($a$), and consumption ($c$). Other variables are reconstructed after solving.

The QZ (generalized Schur) decomposition provides a stable way to separate stable and unstable eigenvalues, enforcing the **Blanchard–Kahn** conditions for a unique saddle-path solution.

### Implementation Roadmap

We now implement the linearized system and solve it with QZ decomposition, then simulate surprise and news shocks to analyze the resulting dynamics.

In [2]:
class RBCModel:
    """
    Implements the canonical Real Business Cycle (RBC) model and solves it using
    the QZ decomposition (Klein's method).
    
    Reduced System Variables: [k_t, a_t, c_t]
    """
    def __init__(self, alpha=0.33, beta=0.99, delta=0.025, sigma=1.0, phi=1.0, rho_A=0.95, sigma_A=0.01, L_ss=1/3):
        self.alpha, self.beta, self.delta, self.sigma, self.phi = alpha, beta, delta, sigma, phi
        self.rho_A, self.sigma_A, self.L_ss_target = rho_A, sigma_A, L_ss
        self.ss = self._solve_steady_state()
        self.solution = self._solve_model()
        
    def _solve_steady_state(self):
        # 1. Solve for r and w/L ratios from FOCs
        r_ss = 1/self.beta - 1 + self.delta
        ky_ratio = self.alpha / r_ss
        
        # 2. Solve for levels given L_ss target
        L_ss = self.L_ss_target
        K_ss = ky_ratio**(1/(1-self.alpha)) * L_ss
        Y_ss = K_ss**self.alpha * L_ss**(1-self.alpha)
        I_ss = self.delta * K_ss
        C_ss = Y_ss - I_ss
        w_ss = (1-self.alpha) * Y_ss / L_ss
        
        # 3. Calibrate psi to support L_ss
        # FOC: psi * C^sigma * L^phi = w
        self.psi = w_ss / (C_ss**self.sigma * L_ss**self.phi)
        
        return pd.Series({'Y': Y_ss, 'K': K_ss, 'L': L_ss, 'C': C_ss, 'I': I_ss, 
                          'w': w_ss, 'r': r_ss, 'psi': self.psi})
    
    def _solve_model(self):
        ss = self.ss
        alpha, beta, delta, sigma = self.alpha, self.beta, self.delta, self.sigma
        phi, rho_A = self.phi, self.rho_A
        r_ss = ss.r
        
        # Reduced System: [k, a, c]
        # k, a are predetermined (states). c is jump.
        n_vars = 3
        n_states = 2
        
        AA = np.zeros((n_vars, n_vars))
        BB = np.zeros((n_vars, n_vars))
        
        # Variable indices: k=0, a=1, c=2
        
        # --- Helper Constants for Substitution ---
        # We linearized l_t out: 
        # l_t = (1/(phi+alpha)) * (a_t + alpha*k_t - sigma*c_t)
        # Let l_t = L_a * a_t + L_k * k_t + L_c * c_t
        denom = phi + alpha
        L_a = 1.0 / denom
        L_k = alpha / denom
        L_c = -sigma / denom
        
        # --- 1. Resource Constraint (Eq for k_{t+1}) ---
        # k_{t+1} = (1-delta)*k_t + (Y_ss/K_ss)*y_t - (C_ss/K_ss)*c_t
        # y_t = a_t + alpha*k_t + (1-alpha)*l_t
        # Substituting l_t:
        # y_t = (1 + (1-alpha)L_a)a_t + (alpha + (1-alpha)L_k)k_t + ((1-alpha)L_c)c_t
        
        Y_K = ss.Y / ss.K
        C_K = ss.C / ss.K
        
        # Coeffs for y_t expansion
        y_a = 1 + (1-alpha)*L_a
        y_k = alpha + (1-alpha)*L_k
        y_c = (1-alpha)*L_c
        
        # k_{t+1} coeff in AA is 1
        AA[0, 0] = 1
        
        # Coeffs in BB
        # k_t terms: (1-delta) + Y_K * y_k
        BB[0, 0] = (1-delta) + Y_K * y_k
        # a_t terms: Y_K * y_a
        BB[0, 1] = Y_K * y_a
        # c_t terms: Y_K * y_c - C_K
        BB[0, 2] = Y_K * y_c - C_K
        
        # --- 2. Euler Equation (Eq for c_{t+1}) ---
        # sigma*c_t = sigma*E_t[c_{t+1}] - beta*r_ss*E_t[r_{t+1}]
        # r_{t+1} = a_{t+1} + (alpha-1)k_{t+1} + (1-alpha)l_{t+1}
        # Substitute l_{t+1}:
        # r_{t+1} = (1 + (1-alpha)L_a)a_{t+1} + (alpha-1 + (1-alpha)L_k)k_{t+1} + ((1-alpha)L_c)c_{t+1}
        
        r_a = 1 + (1-alpha)*L_a
        r_k = (alpha-1) + (1-alpha)*L_k
        r_c = (1-alpha)*L_c
        
        # AA terms (t+1 variables)
        # c_{t+1}: sigma - beta*r_ss*r_c
        AA[1, 2] = sigma - beta * r_ss * r_c
        # k_{t+1}: -beta*r_ss*r_k
        AA[1, 0] = -beta * r_ss * r_k
        # a_{t+1}: -beta*r_ss*r_a
        AA[1, 1] = -beta * r_ss * r_a
        
        # BB terms (t variables)
        # c_t: sigma
        BB[1, 2] = sigma
        
        # --- 3. Technology (Eq for a_{t+1}) ---
        # a_{t+1} = rho * a_t
        AA[2, 1] = 1
        BB[2, 1] = rho_A
        
        # Save substitution constants for simulation reconstruction
        self.subs = {'L_a': L_a, 'L_k': L_k, 'L_c': L_c,
                     'y_a': y_a, 'y_k': y_k, 'y_c': y_c,
                     'r_a': r_a, 'r_k': r_k, 'r_c': r_c}
        
        return solve_qz(AA, BB, n_states)
    
    def reconstruct_variables(self, k, a, c):
        """Reconstructs l, y, i, w, r from the core states."""
        s = self.subs
        l = s['L_a']*a + s['L_k']*k + s['L_c']*c
        y = s['y_a']*a + s['y_k']*k + s['y_c']*c
        # i comes from resource: I*i = Y*y - C*c
        # i = (Y_ss/I_ss)*y - (C_ss/I_ss)*c
        i = (self.ss.Y/self.ss.I)*y - (self.ss.C/self.ss.I)*c
        
        # w = y - l (log linear Cobb Douglas implied this in share form, but check def)
        # w_t = a_t + alpha*k_t - alpha*l_t. 
        # Or w_t = y_t - l_t
        w = y - l
        
        # r = y_t - k_t
        r = s['r_a']*a + s['r_k']*k + s['r_c']*c
        
        return l, y, i, w, r

# Instantiate and Solve
rbc = RBCModel()
print("> **Note:** Model Solved. Steady State Output:")
display(rbc.ss.to_frame().T)

> **Note:** Model Solved. Steady State Output:


,Y,K,L,C,I,w,r,psi
0,1.005109,9.449473,0.333333,0.768872,0.236237,2.02027,0.035101,7.882724


# Summary

This part derived the linearized RBC solution and implemented the QZ decomposition needed for stable policy and transition matrices.
